In [1]:
!pip install pandas-gbq google-auth --quiet

import os
import hashlib
import pandas as pd
import pandas_gbq
import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_dark"

PROJECT_ID = "gdelt-494812"
DATASET_ID = "benin_2025"
KEY_PATH = "gdelt-494812-54aad2c4e931.json"


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
def get_credentials():
    if os.path.exists(KEY_PATH):
        from google.oauth2 import service_account
        return service_account.Credentials.from_service_account_file(KEY_PATH)
    return None

credentials = get_credentials()

In [3]:
os.makedirs("cache/", exist_ok=True)

def charger_donnees(query, force_reload=False):
    query_hash = hashlib.md5(query.encode()).hexdigest()[:8]
    cache_path = f"cache/cache_{query_hash}.csv"
    if os.path.exists(cache_path) and not force_reload:
        df = pd.read_csv(cache_path)
    else:
        df = pandas_gbq.read_gbq(query, project_id=PROJECT_ID, credentials=credentials)
        df.to_csv(cache_path, index=False)
    return df

In [4]:
query_events_head = f"""
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.events_clean`
LIMIT 5
"""

df_events = charger_donnees(query_events_head)
df_events.head()

,GLOBALEVENTID,DATEADDED,SQLDATE,MonthYear,Year,Actor1Name,Actor1CountryCode,Actor1Type1Code,Actor2Name,Actor2CountryCode,...,QuadClass_Label,interaction_type,EventCategory,Actor1Role,Actor2Role,goldstein_category,tone_category,has_international_actor,event_scope,is_significant
0,1221339362,20250118064500,20250118,202501,2025,ECOWAS,WAF,IGO,NaN,NaN,...,Verbal Cooperation,Cooperation,Make Public Statement,Intergovernmental Org,NaN,Neutral,Negative,False,Unknown,True
1,1220702354,20250115101500,20250115,202501,2025,ECOWAS,WAF,IGO,BENIN,BEN,...,Verbal Cooperation,Cooperation,Make Public Statement,Intergovernmental Org,Government,Neutral,Very Negative,False,International,True
2,1274392218,20251115230000,20251115,202511,2025,AFRICA,AFR,NaN,UNITED STATES,USA,...,Verbal Cooperation,Cooperation,Make Public Statement,NaN,NaN,Neutral,Negative,True,Unknown,True
3,1248142398,20250606023000,20250606,202506,2025,BENIN,BEN,GOV,NaN,NaN,...,Verbal Cooperation,Cooperation,Make Public Statement,Government,NaN,Neutral,Negative,False,Domestic,True
4,1250982889,20250709024500,20250709,202507,2025,BENIN,BEN,NaN,NaN,NaN,...,Verbal Cooperation,Cooperation,Make Public Statement,NaN,NaN,Neutral,Positive,False,Domestic,True


In [5]:
query_gkg_head = f"""
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.gkg_clean`
LIMIT 5
"""

df_gkg = charger_donnees(query_gkg_head)
df_gkg.head()

,GKGRECORDID,DATE,SourceCommonName,DocumentIdentifier,V2Themes,V2Locations,V2Persons,V2Organizations,V2Tone,TranslationInfo,...,tone_activity,word_count,tone_category,source_language,is_translated,nb_themes,nb_persons,nb_organizations,nb_locations,is_rich_article
0,20250101003000-152,20250101003000,guardian.ng,https://guardian.ng/news/benin-protests-remark...,"ECON_DEBT,737;WB_1104_MACROECONOMIC_VULNERABIL...",1#Nigeria#NI#NI##10#8#NI#712;1#Benin#BN#BN##9....,"Olushegun Bakari,397;Mohamed Bazoum,921",NaN,"-5.14018691588785,2.33644859813084,7.476635514...",NaN,...,17.757009,200.0,Very Negative,eng,False,30,2,0,18,False
1,20250101003000-707,20250101003000,punchng.com,https://punchng.com/five-motorists-escape-deat...,"WOUND,1604;WOUND,2156;CRISISLEX_C03_WELLBEING_...","1#Benin#BN#BN##9.5#2.25#BN#139;5#Ogun State, O...","Abule Oko,2041;Seni Ogunyemi,341","Enforcement Agency,325;Volvo,1380;Volvo,1499","-9.27536231884058,0.869565217391304,10.1449275...",NaN,...,20.869565,332.0,Very Negative,eng,False,51,2,3,17,True
2,20250101003000-198,20250101003000,punchng.com,https://punchng.com/pdp-seeks-igs-intervention...,"TAX_FNCACT_CHIEF,1138;TAX_FNCACT_CHIEF,2677;TA...","5#Edo State, Edo, Nigeria#NI#NI37#191281#6.5#6...","Justice Daniel Okungbowa,1193;Justice Efe Ikpo...","House Of Assembly,528;House Of Assembly,1542;P...","-11.6822429906542,1.86915887850467,13.55140186...",NaN,...,18.224299,402.0,Very Negative,eng,False,100,6,4,6,True
3,20250101003000-176,20250101003000,punchng.com,https://punchng.com/fg-votes-n365bn-for-new-ro...,"ENV_SOLAR,3642;WB_841_JUSTICE_SYSTEM_ADMINISTR...","4#Shendam, Nigeria (General), Nigeria#NI#NI00#...","Kwanar Dakatsalle-Falgore,1858;Toyota Hilux,27...","National Assembly,2995;Ministry Of Works,29","-0.599700149925037,1.19940029985008,1.79910044...",NaN,...,15.442279,625.0,Neutral,eng,False,82,6,2,21,True
4,20250101004500-259,20250101004500,pakobserver.net,https://pakobserver.net/saudi-fund-for-develop...,"WB_2433_CONFLICT_AND_VIOLENCE,126;WB_2432_FRAG...",1#Saudi Arabia#SA#SA##25#45#SA#656;1#Saudi Ara...,NaN,"Saudi Fund For Development,219","2.9940119760479,4.49101796407186,1.49700598802...",NaN,...,15.868263,292.0,Positive,eng,False,43,0,1,17,False


In [6]:
query_mentions_head = f"""
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.mentions_clean`
LIMIT 5
"""

df_mentions = charger_donnees(query_mentions_head)
df_mentions.head()

,GLOBALEVENTID,MentionTimeDate,MentionType,MentionSourceName,Confidence,MentionDocTone,MentionDocTranslationInfo,mention_datetime,mention_date,mention_year_month,MentionType_Label,tone_category,confidence_level,source_language,is_translated
0,1227182544,20250218050000,1,14ymedio.com,100,1.683502,srclc:spa;eng:GT-SPA 1.0,2025-02-18 05:00:00+00:00,2025-02-18,2025-02,Web,Positive,High,spa,True
1,1227182830,20250218050000,1,14ymedio.com,40,1.683502,srclc:spa;eng:GT-SPA 1.0,2025-02-18 05:00:00+00:00,2025-02-18,2025-02,Web,Positive,Low,spa,True
2,1227182831,20250218050000,1,14ymedio.com,60,1.683502,srclc:spa;eng:GT-SPA 1.0,2025-02-18 05:00:00+00:00,2025-02-18,2025-02,Web,Positive,Medium,spa,True
3,1268569518,20251014160000,1,14ymedio.com,100,3.072626,srclc:spa;eng:GT-SPA 1.0,2025-10-14 16:00:00+00:00,2025-10-14,2025-10,Web,Positive,High,spa,True
4,1268569834,20251014160000,1,14ymedio.com,100,3.072626,srclc:spa;eng:GT-SPA 1.0,2025-10-14 16:00:00+00:00,2025-10-14,2025-10,Web,Positive,High,spa,True


## Question analytique

# Comment évolue le volume d'événements médiatiques au Bénin tout au long de 2025 ?

In [19]:
# Récupère le nombre total d'événements par mois depuis la table events_clean
query_volume_evenements = f"""
SELECT
    MonthYear AS year_month,
    COUNT(*) AS volume_events
FROM `{PROJECT_ID}.{DATASET_ID}.events_clean`
WHERE MonthYear IS NOT NULL
GROUP BY year_month
ORDER BY year_month
"""
# Exécute la requête SQL et charge les résultats dans un DataFrame pandas
df_events_volume = charger_donnees(query_volume_evenements).copy()

In [20]:
# Convertit la colonne mensuelle au format date puis trie les données chronologiquement
df_events_volume["year_month"] = pd.to_datetime(
    df_events_volume["year_month"].astype(str),
    format="%Y%m"
)
df_events_volume = df_events_volume.sort_values("year_month").reset_index(drop=True)
df_events_volume["label_month"] = df_events_volume["year_month"].dt.strftime("%b %Y")

In [21]:
# Identifie le mois le plus actif, le mois le plus calme et la moyenne mensuelle
idx_max = df_events_volume["volume_events"].idxmax()
idx_min = df_events_volume["volume_events"].idxmin()

mois_max = df_events_volume.loc[idx_max, "label_month"]
val_max  = int(df_events_volume.loc[idx_max, "volume_events"])

mois_min = df_events_volume.loc[idx_min, "label_month"]
val_min  = int(df_events_volume.loc[idx_min, "volume_events"])

In [11]:
# Crée une courbe temporelle pour visualiser l'évolution mensuelle du volume d'événements
fig_events = px.line(
    df_events_volume,
    x="year_month",
    y="volume_events",
    markers=True,
    title="Évolution mensuelle du volume d'événements sur le Bénin"
)

In [12]:
# Personnalise la ligne principale, les points, la zone remplie et le survol
fig_events.update_traces(
    line=dict(color="#2F6BFF", width=3),
    marker=dict(size=8, color="#2F6BFF"),
    fill="tozeroy",
    fillcolor="rgba(47,107,255,0.08)",
    hovertemplate="<b>%{x|%b %Y}</b><br>Événements : %{y:,}<extra></extra>"
)

In [ ]:
# Réinitialisons les annotations pour éviter les doublons à chaque ré-exécution
fig_events.layout.annotations = []

# Ajoutons les annotations pour le mois le plus actif et le mois le plus calme
fig_events.add_annotation(
    x=df_events_volume.loc[idx_max, "year_month"],
    y=val_max,
    text=f"Mois le plus actif<br><b>{mois_max}</b><br>{val_max:,} événements",
    showarrow=True, arrowhead=2, ax=0, ay=-85,
    bgcolor="rgba(228,87,86,0.12)", bordercolor="#E45756", borderwidth=1,
    font=dict(size=11)
)

fig_events.add_annotation(
    x=df_events_volume.loc[idx_min, "year_month"],
    y=val_min,
    text=f"Mois le plus calme<br><b>{mois_min}</b><br>{val_min:,} événements",
    showarrow=True, arrowhead=2, ax=0, ay=-85,
    bgcolor="rgba(44,177,161,0.12)", bordercolor="#2CB1A1", borderwidth=1,
    font=dict(size=11)
)

# Affichons le total global comme sous-titre intégré dans le graphique
fig_events.add_annotation(
    text=f"Total annuel : <b>{df_events_volume['volume_events'].sum():,} événements</b>",
    xref="paper", yref="paper",
    x=1.0, y=1.08, xanchor="right", showarrow=False,
    font=dict(size=13, color="#6B7280"),
    bgcolor="rgba(0,0,0,0.04)", bordercolor="rgba(0,0,0,0.08)", borderwidth=1
)

fig_events.show()

## Insights

### **Insight 1 — Décembre : explosion du volume événementiel**

**Décembre concentre 5 784 événements**, soit **18,5% du volume annuel total** (31 504 événements). Ce mois enregistre **deux fois le volume mensuel moyen** (2 625 événements), révélant un **pic d'activité médiatique sans équivalent** sur l'année.

### **Insight 2 — Juin : effondrement brutal du volume**

**Juin enregistre le minimum annuel avec seulement 1 190 événements**, soit **54,7% en dessous de la moyenne mensuelle**. Ce creux représente un **trou médiatique** marqué, contrastant avec les mois adjacents (2 394 en mai, 2 955 en juillet).

### **Insight 3 — Trois périodes distinctes**

L'année se divise en **trois phases** : un **début actif** (janvier-mars : 2 580-2 981 événements), un **creux estival** (juin-octobre : 1 190-2 394 événements), et une **explosion de fin d'année** (décembre). Cette saisonnalité révèle une couverture **événementielle** plutôt que **structurelle**.

### **Insight 4 — Cohérence avec les mentions GDELT**

Le pattern mensuel des événements suit exactement la dynamique des mentions : **pic en décembre** (7 730 mentions), **creux en juin** (1 216 mentions). Cette corrélation confirme que la visibilité du Bénin repose sur des **événements ponctuels** amplifiés par la répétition médiatique.

## Synthèse

Le volume d'événements médiatiques au Bénin en 2025 suit une **trajectoire instable** dominée par **deux anomalies extrêmes** : un **effondrement en juin** (1 190 événements, -55% par rapport à la moyenne) et une **explosion en décembre** (5 784 événements, +120%). Le reste de l'année oscille autour de **2 000-3 000 événements par mois**, avec une légère baisse estivale. Cette volatilité confirme que la visibilité internationale du Bénin est **événementielle** plutôt que continue, dépendant de pics d'actualité ponctuels qui amplifient brutalement la couverture.

## Question analytique

# Quelle est la nature dominante des interactions du Bénin avec le monde en 2025, et cette dynamique évolue-t-elle dans le temps ?

In [13]:
# ── Palette centralisée ─────────────────────────────────────────────
# ── Palette centralisée — Jaune (Coopération) & Rouge (Conflit) ─────
COLOR_COOP_DARK  = "#F5A800"   # Jaune ambré foncé  — Verbal Cooperation
COLOR_COOP_LIGHT = "#FFD166"   # Jaune doux clair   — Material Cooperation
COLOR_CONF_DARK  = "#E45756"   # Rouge vif foncé    — Verbal Conflict
COLOR_CONF_LIGHT = "#FF9E9E"   # Rouge rosé clair   — Material Conflict

# ── Query ───────────────────────────────────────────────────────────
query_donut = f"""
SELECT
    QuadClass_Label,
    interaction_type,
    COUNT(*) AS nb_events
FROM `{PROJECT_ID}.{DATASET_ID}.events_clean`
WHERE QuadClass_Label IS NOT NULL
GROUP BY QuadClass_Label, interaction_type
ORDER BY nb_events DESC
"""
df_donut = charger_donnees(query_donut)

# ── Palette fixe sur les 4 labels GDELT ─────────────────────────────
color_map_donut = {
    "Verbal Cooperation":   COLOR_COOP_DARK,
    "Material Cooperation": COLOR_COOP_LIGHT,
    "Verbal Conflict":      COLOR_CONF_DARK,
    "Material Conflict":    COLOR_CONF_LIGHT
}

# ── KPIs ─────────────────────────────────────────────────────────────
total = df_donut["nb_events"].sum()

# ── Figure ───────────────────────────────────────────────────────────
fig_donut = px.pie(
    df_donut,
    names="QuadClass_Label",
    values="nb_events",
    hole=0.45,
    title="Bénin 2025 : Répartition des types d'événements (Coopération vs Conflit)",
    color="QuadClass_Label",
    color_discrete_map=color_map_donut
)

fig_donut.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Événements : %{value:,}<br>"
        "Part : %{percent:.1%}"
        "<extra></extra>"
    )
)

fig_donut.update_layout(
    template="plotly_dark",
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    annotations=[dict(
        text=f"<b>{total:,}</b><br>événements",
        x=0.5, y=0.5,
        font=dict(size=13, color="white"),
        showarrow=False
    )],
    title=dict(font=dict(size=16)),
    margin=dict(t=60, b=80, l=20, r=20)
)

fig_donut.show()

Downloading: 100%|██████████|


## Visualisation 1 : Répartition globale des types d'événements

### **Insight 1 — La coopération domine largement**

**23 502 événements de coopération** contre **8 002 événements de conflit**, soit un **ratio de 2,9:1**. Les trois quarts de la couverture événementielle du Bénin concernent des interactions coopératives, reflétant une image internationale majoritairement collaborative.

### **Insight 2 — La coopération est surtout verbale**

**86,5% des événements de coopération** sont de type "Verbal Cooperation" (20 337 événements). La coopération matérielle ne représente que **13,5%** (3 165 événements). Le Bénin génère plus de **déclarations et engagements** que de réalisations concrètes visibles dans les médias.

### **Insight 3 — Le conflit est matériel et verbal à parts égales**

Le conflit se répartit entre **Material Conflict** (54,4%, 4 354 événements) et **Verbal Conflict** (45,6%, 3 648 événements). Contrairement à la coopération, les conflits impliquent autant d'**actions concrètes** que de tensions diplomatiques ou verbales.

In [14]:
# Question : évolution coopération vs conflit dans le temps
query_coop_vs_conflit = f"""
SELECT
    MonthYear AS year_month,
    interaction_type,
    COUNT(*) AS nb_events,
    AVG(GoldsteinScale) AS score_goldstein_moyen
FROM `{PROJECT_ID}.{DATASET_ID}.events_clean`
WHERE MonthYear IS NOT NULL
GROUP BY year_month, interaction_type
ORDER BY year_month
"""
df_coop = charger_donnees(query_coop_vs_conflit)
df_coop["year_month"] = pd.to_datetime(df_coop["year_month"].astype(str), format="%Y%m")

fig_coop = px.line(
    df_coop,
    x="year_month",
    y="nb_events",
    color="interaction_type",
    markers=True,
    title="Coopération vs Conflit au Bénin — Évolution 2025",
    color_discrete_map={
        "Cooperation": "#2F6BFF",   # bleu
        "Conflict":    "#E45756"    # rouge
    }
)
fig_coop.update_traces(line=dict(width=2.5), marker=dict(size=7))
fig_coop.update_layout(
    template="plotly_dark",
    legend_title_text="Type d'interaction",
    xaxis_title="Mois",
    yaxis_title="Nombre d'événements",
    hovermode="x unified"
)
fig_coop.show()

Downloading: 100%|██████████|


## Visualisation 2 : Évolution mensuelle Coopération vs Conflit

### **Insight 1 — La coopération domine chaque mois sans exception**

Sur les **12 mois de 2025**, la coopération dépasse toujours le conflit, avec un **ratio minimal de 2:1** (juin) et un **pic à 3,1:1** (avril). Aucun mois ne présente une inversion, confirmant une **structure stable** de prédominance coopérative.

### **Insight 2 — Décembre : explosion simultanée des deux types**

**Décembre concentre 3 953 événements de coopération et 1 831 de conflit**, soit respectivement **26% et 23% des totaux annuels**. Ce mois capte à lui seul **près d'un quart de l'activité médiatique**, tant coopérative que conflictuelle, signalant un **événement majeur multi-dimensionnel**.

### **Insight 3 — Juin : creux minimal pour les deux dynamiques**

**Juin enregistre les volumes les plus bas** : 891 coopérations et 299 conflits. Ce mois représente un **trou médiatique généralisé**, affectant proportionnellement les deux types d'interactions. Le ratio reste cependant stable (3:1).

### **Insight 4 — Le score Goldstein confirme la polarité**

La coopération affiche un **Goldstein moyen de +2,8 à +3,3** (impact positif), tandis que le conflit oscille entre **-5,5 et -6,8** (impact négatif). Cette stabilité des scores sur 12 mois indique que la **nature des événements reste constante**, seul le volume varie.

## Synthèse globale : Coopération vs Conflit 2025

La couverture médiatique du Bénin en 2025 est **structurellement coopérative** : **74,6% d'événements de coopération** (ratio 2,9:1), avec une prédominance de la **coopération verbale** (86,5% des coopérations). Cette dynamique est **stable dans le temps**, le ratio mensuel Coopération/Conflit ne descendant jamais sous 2:1. Cependant, la coopération reste largement **déclarative** plutôt que matérielle, tandis que le conflit se traduit autant par des **actions concrètes** que par des tensions verbales. Décembre 2025 marque une **anomalie majeure** avec une explosion simultanée des deux dynamiques, révélant un événement complexe combinant à la fois coopération et conflit à grande échelle.

# Quel est le ton dominant de la couverture médiatique du Bénin en 2025 ?


In [12]:
# — Ton médiatique mensuel depuis gkg_clean
query_tone = f"""
SELECT
  DATE_TRUNC(PARSE_DATE('%Y%m%d', SUBSTR(CAST(DATE AS STRING), 1, 8)), MONTH) AS year_month,
  AVG(CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64)) AS tone_moyen,
  COUNT(*) AS nb_articles
FROM `{PROJECT_ID}.{DATASET_ID}.gkg_clean`
WHERE V2Tone IS NOT NULL
  AND DATE IS NOT NULL
GROUP BY year_month
ORDER BY year_month
"""

df_tone = charger_donnees(query_tone).copy()

Downloading: 100%|██████████|


In [17]:
df_tone['year_month'] = pd.to_datetime(df_tone['year_month'])
df_tone = df_tone.sort_values('year_month').reset_index(drop=True)
df_tone['label_month'] = df_tone['year_month'].dt.strftime('%b %Y')

# Identifier les extremes
idx_min = df_tone['tone_moyen'].idxmin()
idx_max = df_tone['tone_moyen'].idxmax()

mois_negatif = df_tone.loc[idx_min, 'label_month']
val_negatif = round(df_tone.loc[idx_min, 'tone_moyen'], 2)

mois_positif = df_tone.loc[idx_max, 'label_month']
val_positif = round(df_tone.loc[idx_max, 'tone_moyen'], 2)

tone_moyen_global = round(df_tone['tone_moyen'].mean(), 2)

In [18]:
COLOR_NEG = '#E45756'
COLOR_POS = '#2CB1A1'
COLOR_LINE = '#2F6BFF'

fig_tone = px.line(
    df_tone,
    x='year_month',
    y='tone_moyen',
    markers=True,
    title='Évolution mensuelle du ton médiatique autour du Bénin — 2025'
)

fig_tone.update_traces(
    line=dict(color=COLOR_LINE, width=3),
    marker=dict(size=8, color=COLOR_LINE),
    hovertemplate='<b>%{x|%b %Y}</b><br>Ton moyen : %{y:.2f}<extra></extra>'
)

# Ligne de référence à 0 (neutre)
fig_tone.add_hline(
    y=0,
    line_dash='dash',
    line_color='rgba(255,255,255,0.3)',
    annotation_text='Ton neutre (0)',
    annotation_position='bottom right',
    annotation_font_size=11
)

# Annotation mois le plus négatif
fig_tone.add_annotation(
    x=df_tone.loc[idx_min, 'year_month'],
    y=val_negatif,
    text=f'Mois le plus négatif<br><b>{mois_negatif}</b><br>{val_negatif}',
    showarrow=True, arrowhead=2, ax=0, ay=-70,
    bgcolor='rgba(228,87,86,0.12)', bordercolor=COLOR_NEG,
    borderwidth=1, font=dict(size=11)
)

# Annotation mois le plus positif
fig_tone.add_annotation(
    x=df_tone.loc[idx_max, 'year_month'],
    y=val_positif,
    text=f'Mois le plus positif<br><b>{mois_positif}</b><br>{val_positif}',
    showarrow=True, arrowhead=2, ax=0, ay=-70,
    bgcolor='rgba(44,177,161,0.12)', bordercolor=COLOR_POS,
    borderwidth=1, font=dict(size=11)
)

# Moyenne globale en sous-titre
fig_tone.add_annotation(
    text=f'Ton moyen global 2025 : <b>{tone_moyen_global}</b>',
    xref='paper', yref='paper',
    x=1.0, y=1.08, xanchor='right',
    showarrow=False,
    font=dict(size=13, color='#6B7280'),
    bgcolor='rgba(0,0,0,0.04)', bordercolor='rgba(0,0,0,0.08)', borderwidth=1
)

fig_tone.update_layout(
    template='plotly_dark',
    xaxis_title='Mois',
    yaxis_title='Ton moyen (V2Tone)',
    legend=dict(tracegroupgap=0)
)

fig_tone.show()

## Visualisation 1 : Évolution mensuelle du ton

### **Insight 1 — Un ton globalement négatif toute l'année**

Le **ton moyen annuel est de -0.52**, indiquant une couverture médiatique majoritairement négative. Sur 12 mois, **11 mois affichent un ton négatif**, seul novembre étant légèrement positif (+0.41). La perception médiatique du Bénin est structurellement pessimiste.

### **Insight 2 — Décembre 2025 : le mois le plus négatif**

**Décembre enregistre un ton de -1.71**, le plus bas de l'année, associé à un **pic de 8 359 articles**. Ce mois cumule donc **volume record et ton extrêmement négatif**, signalant un événement majeur à forte charge émotionnelle négative.

### **Insight 3 — Juin : double chute (volume et ton)**

**Juin combine le minimum de volume** (2 082 articles) **et le deuxième ton le plus négatif** (-1.49). Ce mois représente un **trou médiatique négatif**, suggérant soit un manque d'actualités, soit une couverture focalisée sur des événements très négatifs.

### **Insight 4 — Novembre : seule exception positive**

**Novembre est le seul mois positif** (+0.41) de l'année. Avec 3 277 articles, ce mois marque une **rupture tonale**, probablement liée à des événements politiques, économiques ou diplomatiques perçus favorablement.

In [ ]:
# ── Palette pour le ton ─────────────────────────────────────────
COLOR_VERY_NEG = "#8B0000"    
COLOR_NEG      = "#E45756"    
COLOR_NEUTRAL  = "#A6ACAF"    
COLOR_POS      = "#2CB1A1"     
COLOR_VERY_POS = "#00C853"  

# ── Query — Distribution des sentiments (donut) ──────────────────
query_tone_distrib = f"""
SELECT
    tone_category,
    COUNT(*) AS nb_articles
FROM `{PROJECT_ID}.{DATASET_ID}.gkg_clean`
WHERE tone_category IS NOT NULL
GROUP BY tone_category
ORDER BY nb_articles DESC
"""
df_tone_distrib = charger_donnees(query_tone_distrib)

# ── Palette fixe sur les 5 catégories de ton ────────────────────
color_map_tone = {
    "Very Negative": COLOR_VERY_NEG,
    "Negative":      COLOR_NEG,
    "Neutral":       COLOR_NEUTRAL,
    "Positive":      COLOR_POS,
    "Very Positive": COLOR_VERY_POS
}

# ── KPIs ──────────────────────────────────────────────────────────
total_articles = df_tone_distrib["nb_articles"].sum()

# ── Figure ────────────────────────────────────────────────────────
fig_tone = px.pie(
    df_tone_distrib,
    names="tone_category",
    values="nb_articles",
    hole=0.45,
    title="Bénin 2025 : Distribution du ton médiatique (V2Tone)",
    color="tone_category",
    color_discrete_map=color_map_tone,
    category_orders={"tone_category": ["Very Negative", "Negative", "Neutral", "Positive", "Very Positive"]}
)

fig_tone.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Articles : %{value:,}<br>"
        "Part : %{percent:.1%}"
        "<extra></extra>"
    )
)

fig_tone.update_layout(
    template="plotly_dark",
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    annotations=[dict(
        text=f"<b>{total_articles:,}</b><br>articles",
        x=0.5, y=0.5,
        font=dict(size=13, color="white"),
        showarrow=False
    )],
    title=dict(font=dict(size=16)),
    margin=dict(t=60, b=80, l=20, r=20)
)

fig_tone.show()

## Visualisation 2 : Distribution des catégories de ton

### **Insight 1 — Domination du ton négatif et très négatif**

**59,2% des articles** sont classés "Negative" ou "Very Negative". Plus de la moitié de la couverture du Bénin porte une **charge émotionnelle négative marquée**, confirmant le biais pessimiste de la médiatisation internationale.

### **Insight 2 — Le ton neutre est minoritaire**

Seulement **23,7% des articles** sont neutres. La couverture médiatique du Bénin est donc **polarisée** : elle penche massivement vers le négatif, avec peu de place pour un traitement factuel et équilibré.

### **Insight 3 — Le ton positif est marginal**

**Moins de 17% des articles** sont "Positive" ou "Very Positive". La couverture favorable du Bénin reste **exceptionnelle**, révélant une difficulté structurelle du pays à générer des récits médiatiques positifs à l'international.

## Synthèse globale : Ton médiatique 2025

La couverture médiatique du Bénin en 2025 est **structurellement négative** : **ton moyen annuel de -0.52**, avec **11 mois sur 12 négatifs** et **59% des articles classés négatifs ou très négatifs**. Les deux crises majeures de l'année sont **juin** (ton -1.49, volume minimal) et **décembre** (ton -1.71, volume maximal). Novembre reste la seule exception positive (+0.41), mais elle ne compense pas la tendance de fond. Cette double perspective (évolution temporelle + distribution des catégories) confirme que la visibilité internationale du Bénin repose davantage sur des **événements négatifs** que sur des récits de développement, coopération ou réussite.

## Question analytique

# Quelle est la distribution géographique des événements médiatiques au Bénin en 2025 ?

In [9]:
# VIZ 4 — Cartographie des événements GDELT au Bénin 2025
query_carte = f"""
SELECT
  ActionGeo_FullName    AS lieu,
  ActionGeo_Lat         AS lat,
  ActionGeo_Long        AS lon,
  interaction_type,
  COUNT(*)              AS nb_events,
  AVG(GoldsteinScale)   AS goldstein_moyen
FROM `{PROJECT_ID}.{DATASET_ID}.events_clean`
WHERE
  ActionGeo_CountryCode = 'BN'
  AND ActionGeo_Lat  IS NOT NULL
  AND ActionGeo_Long IS NOT NULL
  AND CAST(Year AS INT64) = 2025
GROUP BY lieu, lat, lon, interaction_type
ORDER BY nb_events DESC
"""

df_carte = charger_donnees(query_carte).copy()

Downloading: 100%|██████████|


In [11]:
# Installation de la bibliothèque Folium pour la cartographie interactive
!pip install folium --quiet

import folium
from folium.plugins import HeatMap
import pandas as pd

# PRÉPARATION DES DONNÉES

# Agrégation des données par lieu géographique
df_map = (
    df_carte
    .groupby(["lieu", "lat", "lon"])
    .agg(
        nb_events        = ("nb_events",        "sum"),
        interaction_type = ("interaction_type", lambda x: x.mode()[0]),
        goldstein_moyen  = ("goldstein_moyen",  "mean"),
    )
    .reset_index()
    .sort_values("nb_events", ascending=False)
)

# Définition des couleurs selon le type d'interaction
COLOR_COOP = "#F5A800"  # Jaune pour la coopération
COLOR_CONF = "#E45756"  # Rouge pour le conflit

df_map["color"] = df_map["interaction_type"].map(
    {"Cooperation": COLOR_COOP, "Conflict": COLOR_CONF}
).fillna("#888888")

# CRÉATION DE LA CARTE DE BASE

# Carte centrée sur le Bénin avec zoom initial
m = folium.Map(
    location=[9.3, 2.3],
    zoom_start=8,
    tiles="OpenStreetMap",
    prefer_canvas=True,
    zoom_control=True,
    scrollWheelZoom=True,
)

# HEATMAP : DENSITÉ SPATIALE DES ÉVÉNEMENTS

# Préparation des données pour la heatmap
heat_data = [
    [row["lat"], row["lon"], row["nb_events"]]
    for _, row in df_map.iterrows()
]

# Ajout de la couche heatmap avec gradient de couleur
HeatMap(
    heat_data,
    radius=18,
    blur=15,
    min_opacity=0.3,
    max_zoom=10,
    gradient={"0.3": "#F5A800", "0.6": "#ff6b35", "1.0": "#E45756"},
).add_to(m)

# MARQUEURS CIRCULAIRES : POINTS INDIVIDUELS

for _, row in df_map.iterrows():
    # Taille du cercle proportionnelle au nombre d'événements
    radius = max(5, min(row["nb_events"] * 0.5, 35))
    
    # Symbole selon le type d'interaction
    symbol = "Cooperation" if row["interaction_type"] == "Cooperation" else "Conflict"
    
    # Contenu du popup
    popup_html = f"""
    <div style="font-family:Inter,sans-serif;min-width:180px;">
        <b style="font-size:14px;">{row['lieu'].split(',')[0]}</b><br>
        <hr style="margin:4px 0;border-color:#eee;">
        <b>{row['interaction_type']}</b><br>
        <b>{int(row['nb_events'])}</b> événements<br>
        Goldstein : <b>{round(row['goldstein_moyen'], 2)}</b>
    </div>
    """
    
    # Ajout du marqueur circulaire
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=radius,
        color="white",
        weight=1.2,
        fill=True,
        fill_color=row["color"],
        fill_opacity=0.82,
        popup=folium.Popup(popup_html, max_width=240),
        tooltip=f"{row['lieu'].split(',')[0]} - {int(row['nb_events'])} évts",
    ).add_to(m)

# LÉGENDE ET TABLEAU STATISTIQUE PAR RÉGION

# Séparation des données : centroïde national vs villes réelles
CENTROID_LAT, CENTROID_LON = 9.5, 2.25

df_centroid = df_carte[
    (df_carte["lat"] == CENTROID_LAT) & (df_carte["lon"] == CENTROID_LON)
]
df_villes = df_carte[
    ~((df_carte["lat"] == CENTROID_LAT) & (df_carte["lon"] == CENTROID_LON))
]

# Statistiques pour le centroïde national
centroid_total = int(df_centroid["nb_events"].sum())
centroid_conf  = int(df_centroid[df_centroid["interaction_type"] == "Conflict"]["nb_events"].sum())
centroid_coop  = int(df_centroid[df_centroid["interaction_type"] == "Cooperation"]["nb_events"].sum())
centroid_pct_c = round(centroid_conf / centroid_total * 100, 1) if centroid_total > 0 else 0
centroid_pct_k = round(centroid_coop / centroid_total * 100, 1) if centroid_total > 0 else 0

# Classification par région géographique
df_region = df_villes.copy()
df_region["region"] = df_region["lat"].apply(
    lambda x: "Nord (>9°N)" if x > 9 else ("Centre (7–9°N)" if x >= 7 else "Sud (<7°N)")
)

# Tableau croisé par région et type d'interaction
region_pivot = df_region.pivot_table(
    index="region", columns="interaction_type",
    values="nb_events", aggfunc="sum"
).fillna(0)

region_pivot["total"]    = region_pivot.sum(axis=1)
region_pivot["pct_conf"] = (region_pivot.get("Conflict", 0) / region_pivot["total"] * 100).round(1)
region_pivot["pct_coop"] = (region_pivot.get("Cooperation", 0) / region_pivot["total"] * 100).round(1)

# Génération des lignes HTML pour le tableau
region_rows_html = ""
for region in ["Nord (>9°N)", "Centre (7–9°N)", "Sud (<7°N)"]:
    if region in region_pivot.index:
        r = region_pivot.loc[region]
        total = int(r["total"])
        pct_c = r["pct_conf"]
        pct_k = r["pct_coop"]
        bar_color = "#E45756" if pct_c > 40 else ("#ff6b35" if pct_c > 20 else "#F5A800")
        region_rows_html += f"""
        <tr>
            <td style="padding:4px 6px;font-size:11px;font-weight:600;">{region}</td>
            <td style="padding:4px 6px;font-size:11px;text-align:center;">{total}</td>
            <td style="padding:4px 6px;font-size:11px;text-align:center;color:{bar_color};font-weight:700;">{pct_c}%</td>
            <td style="padding:4px 6px;font-size:11px;text-align:center;color:#F5A800;font-weight:700;">{pct_k}%</td>
        </tr>"""

# Ligne pour le centroïde national
region_rows_html += f"""
        <tr style="background:#f9fafb;border-top:1px dashed #d1d5db;">
            <td style="padding:4px 6px;font-size:11px;font-style:italic;color:#6b7280;">Bénin générique</td>
            <td style="padding:4px 6px;font-size:11px;text-align:center;color:#6b7280;">{centroid_total}</td>
            <td style="padding:4px 6px;font-size:11px;text-align:center;color:#9ca3af;">{centroid_pct_c}%</td>
            <td style="padding:4px 6px;font-size:11px;text-align:center;color:#9ca3af;">{centroid_pct_k}%</td>
        </tr>"""

# HTML de la légende complète
legend_html = f"""
<div style="
    position:fixed; bottom:30px; left:30px; z-index:9999;
    background:rgba(255,255,255,0.97); color:#1f2937;
    padding:14px 18px; border-radius:12px;
    font-family:Inter,sans-serif; font-size:13px;
    border:1px solid #e5e7eb; box-shadow:0 4px 20px rgba(0,0,0,0.15);
    max-width:280px;
">
    <b style="font-size:14px;display:block;margin-bottom:8px;">
        Bénin - Événements 2025
    </b>

    <span style="color:#F5A800;font-size:18px;">●</span>&nbsp;Coopération<br>
    <span style="color:#E45756;font-size:18px;">●</span>&nbsp;Conflit<br>

    <div style="margin-top:8px;padding-top:8px;border-top:1px solid #e5e7eb;
                color:#6b7280;font-size:11px;line-height:1.5;">
        Taille du cercle = volume d'événements<br>
        Zone colorée = densité (heatmap)<br>
        Cliquer sur un point pour les détails
    </div>

    <div style="margin-top:10px;padding-top:10px;border-top:1px solid #e5e7eb;">
        <b style="font-size:11px;color:#374151;">Répartition par zone géographique</b>
        <table style="width:100%;margin-top:6px;border-collapse:collapse;">
            <thead>
                <tr style="background:#f9fafb;">
                    <th style="padding:4px 6px;font-size:10px;text-align:left;color:#6b7280;">Région</th>
                    <th style="padding:4px 6px;font-size:10px;text-align:center;color:#6b7280;">Évts</th>
                    <th style="padding:4px 6px;font-size:10px;text-align:center;color:#E45756;">Conflit</th>
                    <th style="padding:4px 6px;font-size:10px;text-align:center;color:#F5A800;">Coop.</th>
                </tr>
            </thead>
            <tbody>
                {region_rows_html}
            </tbody>
        </table>
        <div style="margin-top:5px;font-size:9px;color:#9ca3af;font-style:italic;">
            Note : Bénin générique = articles sans ville précise,<br>
            géolocalisés au centroïde national (9.5°N, 2.25°E)
        </div>
    </div>
</div>
"""

# Ajout de la légende à la carte
m.get_root().html.add_child(folium.Element(legend_html))

# EXPORT DE LA CARTE

m.save("viz4_carte_benin_2025.html")
print("Carte exportée avec succès : viz4_carte_benin_2025.html")


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Carte exportée avec succès : viz4_carte_benin_2025.html


## Insights

### **Insight 1 — Domination du centroïde national**

**21 758 événements sur 23 308** (93,3%) sont géolocalisés au point générique "Benin" (9.5°N, 2.25°E). Seulement **6,7% des événements** sont associés à des villes précises. La majorité de la couverture médiatique manque de granularité géographique, révélant une **limite structurelle** des données GDELT pour le Bénin.

### **Insight 2 — Les villes réelles concentrent 1 550 événements**

Parmi les événements géolocalisés avec précision, **Ouidah (218), Porto-Novo (196), Abomey (155) et Parakou (110)** dominent. Ces quatre villes concentrent **48,6% du volume urbain**. La visibilité médiatique précise se concentre dans les pôles administratifs et économiques.

### **Insight 3 — Le Nord est majoritairement conflictuel**

Les départements d'**Alibori et Atakora** (Nord, >9°N) affichent un ratio **Conflit/Coopération très élevé** : Alibori avec 73 conflits vs 36 coopérations, Kandi avec 59 conflits vs 42 coopérations. Le Nord capte une couverture médiatique **structurée par l'insécurité**, tandis que le Sud affiche un profil plus équilibré.

### **Insight 4 — Coopération massive au centroïde national**

Au point générique "Benin", **16 081 événements de coopération** contre **5 677 de conflit** (ratio 2,8:1). Ce contraste avec le profil conflictuel du Nord suggère que les événements non géolocalisés précisément concernent davantage des **affaires institutionnelles, diplomatiques ou économiques** que des événements sécuritaires.

## Synthèse

La géographie médiatique du Bénin en 2025 révèle **deux limites critiques** : d'abord, **93% des événements** manquent de géolocalisation précise et sont attribués au centroïde national. Ensuite, parmi les **1 550 événements géolocalisés**, la distribution spatiale est **inégale** : le Sud urbain (Ouidah, Porto-Novo, Abomey) concentre les volumes, tandis que le Nord (Alibori, Atakora) capte une couverture **structurée par le conflit**. Cette carte montre autant la **géographie des événements** que les **angles morts** du système de détection GDELT.

## Question analytique

# Comment évolue la visibilité médiatique du Bénin tout au long de l'année 2025 ?

In [6]:
# Évolution mensuelle des mentions GDELT au Bénin en 2025
query_mentions_mois = f"""
SELECT
  m.mention_year_month AS mois,
  COUNT(*)             AS nb_mentions
FROM `{PROJECT_ID}.{DATASET_ID}.mentions_clean` AS m
JOIN `{PROJECT_ID}.{DATASET_ID}.events_clean`   AS e
  ON m.GLOBALEVENTID = e.GLOBALEVENTID
WHERE
  e.ActionGeo_CountryCode = 'BN'
  AND CAST(e.Year AS INT64) = 2025
  AND m.mention_year_month IS NOT NULL
GROUP BY mois
ORDER BY mois ASC
"""

df_mentions_mois = charger_donnees(query_mentions_mois).copy()

Downloading: 100%|██████████|


In [7]:
import plotly.graph_objects as go

mapping_mois = {
    "2025-01": "Jan", "2025-02": "Fév", "2025-03": "Mar", "2025-04": "Avr",
    "2025-05": "Mai", "2025-06": "Jun", "2025-07": "Jul", "2025-08": "Aoû",
    "2025-09": "Sep", "2025-10": "Oct", "2025-11": "Nov", "2025-12": "Déc"
}

df_mentions_mois["mois_label"] = df_mentions_mois["mois"].map(mapping_mois)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_mentions_mois["mois_label"],
    y=df_mentions_mois["nb_mentions"],
    mode="lines+markers",
    line=dict(
        color="#4f98a3",
        width=3,
        shape="spline",
        smoothing=0.9
    ),
    marker=dict(size=8, color="#4f98a3"),
    fill="tozeroy",
    fillcolor="rgba(79,152,163,0.12)",
    hovertemplate="%{x} : <b>%{y:,}</b> mentions<extra></extra>"
))

fig.update_layout(
    title=dict(
        text="Évolution mensuelle des mentions GDELT (Nombre de mentions GDELT par mois · Bénin 2025)",
        x=0.02,
        xanchor="left",
        y=0.97,
        yanchor="top",
        font=dict(size=18, color="#b8c0c8")
    ),
    template="plotly_dark",
    showlegend=False,
    xaxis_title="",
    yaxis_title="Nombre de mentions",
    margin=dict(t=110, l=70, r=40, b=70)
)

fig.add_annotation(
    text="<b>Série temporelle</b>",
    xref="paper", yref="paper",
    x=0.98, y=1.08,
    showarrow=False,
    xanchor="right",
    yanchor="middle",
    font=dict(size=12, color="#4f98a3"),
    bgcolor="rgba(79,152,163,0.12)",
    bordercolor="#4f98a3",
    borderwidth=1,
    borderpad=6
)

fig.show()

### **Insight 1 — Un pic massif en décembre**

**Décembre concentre 7 730 mentions**, soit **24% du volume annuel total**. Ce mois représente à lui seul **plus du double** de la moyenne mensuelle (2 692 mentions). Un événement majeur ou une concentration d'actualités a clairement marqué la fin d'année.

### **Insight 2 — Un creux marqué en juin**

**Juin enregistre le minimum annuel avec seulement 1 216 mentions**, soit **moins de la moitié de la moyenne mensuelle**. Cette chute brutale (−40% par rapport à mai, −60% par rapport à juillet) révèle un **trou médiatique** en milieu d'année.

### **Insight 3 — Une tendance stable avec deux anomalies**

En excluant juin (minimum) et décembre (maximum), la visibilité reste **relativement stable** entre **1 759 et 3 042 mentions par mois**. La couverture du Bénin suit donc un **rythme constant**, ponctué par deux **anomalies extrêmes**.

## Synthèse

La couverture médiatique du Bénin en 2025 suit une **trajectoire stable** (moyenne mensuelle de 2 692 mentions) avec **deux anomalies majeures** : un **effondrement en juin** (1 216 mentions, −55% par rapport à la moyenne) et une **explosion en décembre** (7 730 mentions, +187%). Ces variations suggèrent que la visibilité du pays est **événementielle** plutôt que structurelle, avec des pics liés à des actualités spécifiques plutôt qu'à une couverture continue.

## Question analytique

# Quelle langue domine la couverture médiatique internationale du Bénin en 2025 ?

In [ ]:
#  Langue des sources des mentions GDELT au Bénin en 2025
query_langues = f"""
SELECT
  m.source_language AS langue,
  COUNT(*)          AS nb_mentions
FROM `{PROJECT_ID}.{DATASET_ID}.mentions_clean` AS m
JOIN `{PROJECT_ID}.{DATASET_ID}.events_clean`   AS e
  ON m.GLOBALEVENTID = e.GLOBALEVENTID
WHERE
  e.ActionGeo_CountryCode = 'BN'
  AND CAST(e.Year AS INT64) = 2025
  AND m.source_language IS NOT NULL
GROUP BY langue
ORDER BY nb_mentions DESC
LIMIT 5
"""

df_langues = charger_donnees(query_langues).copy()

Downloading: 100%|██████████|


In [5]:
import plotly.graph_objects as go

# Calcul des pourcentages
df_langues["pct"] = (
    df_langues["nb_mentions"] / df_langues["nb_mentions"].sum() * 100
).round(1)

# Ordre décroissant
df_langues = df_langues.sort_values("pct", ascending=False)

# Une couleur différente par barre
bar_colors = ["#4f98a3", "#e8af34", "#6daa45", "#fdab43", "#a86fdf"]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_langues["langue"],
    y=df_langues["pct"],
    marker_color=bar_colors[:len(df_langues)],
    marker_line_color="rgba(255,255,255,0.10)",
    marker_line_width=1,
    text=df_langues["pct"].astype(str) + "%",
    textposition="outside",
    hovertemplate="%{x} : <b>%{y}%</b><extra></extra>"
))

fig.update_layout(
    title=dict(
        text="Langue des sources (source_language — top 5 langues)",
        x=0.02,
        xanchor="left",
        y=0.97,
        yanchor="top",
        font=dict(size=18, color="#b8c0c8")
    ),
    template="plotly_dark",
    showlegend=False,
    xaxis_title="",
    yaxis_title="% des mentions",
    margin=dict(t=110, l=70, r=40, b=70),
    width=1100,
    height=520,
    bargap=0.22,
    barcornerradius=12
)

fig.add_annotation(
    text="<b>Bar chart</b>",
    xref="paper", yref="paper",
    x=0.98, y=1.08,
    showarrow=False,
    xanchor="right",
    yanchor="middle",
    font=dict(size=12, color="#4f98a3"),
    bgcolor="rgba(79,152,163,0.12)",
    bordercolor="#4f98a3",
    borderwidth=1,
    borderpad=6
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zeroline=False
)

fig.update_xaxes(showgrid=False)

fig.show()

## Insights

### **Insight 1 — L'anglais domine massivement la couverture**

**79,8% des mentions** (25 782 sur 32 308) proviennent de sources en anglais. La visibilité internationale du Bénin est donc **écrasamment structurée par l'écosystème médiatique anglophone**, reflétant la domination des médias anglo-saxons dans le système d'information mondial.

### **Insight 2 — Le français reste significatif mais minoritaire**

**14,9% des mentions** (4 825) sont en français. Malgré le statut du français comme langue officielle du Bénin, la couverture francophone ne représente qu'un septième du volume anglophone, révélant une **asymétrie linguistique** dans la visibilité médiatique du pays.

### **Insight 3 — Les autres langues sont marginales**

Les mentions en espagnol (1,8%), portugais (0,6%) et italien (0,6%) sont **résiduelles**. Ensemble, elles ne représentent que **3,8% du total**. La présence médiatique du Bénin en dehors de l'anglais et du français est quasi inexistante.

## Synthèse

La couverture médiatique du Bénin en 2025 est **linguistiquement polarisée** : **80% en anglais, 15% en français, et moins de 4% dans toutes les autres langues combinées**. Cette répartition reflète la **domination de l'écosystème médiatique anglophone** dans la circulation mondiale de l'information. Le Bénin, bien que francophone, est principalement visible à travers le prisme des médias anglo-saxons, ce qui peut influencer le cadrage et l'interprétation des événements le concernant.

## Question analytique

# Quel est le niveau de fiabilité des mentions médiatiques sur le Bénin en 2025 ?

In [ ]:
#  Niveau de confiance des mentions GDELT au Bénin en 2025
query_confidence = f"""
SELECT
  m.confidence_level AS confidence_cat,
  COUNT(*)           AS nb_mentions
FROM `{PROJECT_ID}.{DATASET_ID}.mentions_clean` AS m
JOIN `{PROJECT_ID}.{DATASET_ID}.events_clean`   AS e
  ON m.GLOBALEVENTID = e.GLOBALEVENTID
WHERE
  e.ActionGeo_CountryCode = 'BN'
  AND CAST(e.Year AS INT64) = 2025
  AND m.confidence_level IS NOT NULL
GROUP BY confidence_cat
ORDER BY nb_mentions DESC
"""

df_confidence = charger_donnees(query_confidence).copy()

Downloading: 100%|██████████|


In [27]:
import plotly.graph_objects as go

# Calcul des pourcentages
df_confidence["pct"] = (
    df_confidence["nb_mentions"] / df_confidence["nb_mentions"].sum() * 100
).round(1)

# Harmoniser les labels si besoin
df_confidence["confidence_cat"] = df_confidence["confidence_cat"].str.title()

# Couleurs : vert (High), jaune (Medium), rouge (Low)
colors_confidence = {
    "High": "rgba(109,170,69,1.0)",    # vert
    "Medium": "rgba(232,175,52,1.0)",  # jaune
    "Low": "rgba(209,99,99,1.0)"       # rouge
}

# Trier par pct décroissant
df_confidence = df_confidence.sort_values("pct", ascending=False)

# Appliquer les couleurs dans l'ordre trié
bar_colors = [colors_confidence.get(c, "#b8c0c8") for c in df_confidence["confidence_cat"]]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_confidence["confidence_cat"],
    y=df_confidence["pct"],
    marker_color=bar_colors,
    marker_line_color="rgba(255,255,255,0.10)",
    marker_line_width=1,
    text=df_confidence["pct"].astype(str) + "%",
    textposition="outside",
    hovertemplate="%{x} : <b>%{y}%</b><extra></extra>"
))

fig.update_layout(
    title=dict(
        text="Niveau de confiance (confidence_level par catégorie)",
        x=0.02,
        xanchor="left",
        y=0.97,
        yanchor="top",
        font=dict(size=18, color="#b8c0c8")
    ),
    template="plotly_dark",
    showlegend=False,
    xaxis_title="",
    yaxis_title="% des mentions",
    margin=dict(t=110, l=70, r=40, b=70),
    width=1100,
    height=520,
    bargap=0.22,
    barcornerradius=12
)

fig.add_annotation(
    text="<b>Bar chart</b>",
    xref="paper", yref="paper",
    x=0.98, y=1.08,
    showarrow=False,
    xanchor="right",
    yanchor="middle",
    font=dict(size=12, color="#4f98a3"),
    bgcolor="rgba(79,152,163,0.12)",
    bordercolor="#4f98a3",
    borderwidth=1,
    borderpad=6
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zeroline=False,
    range=[0, df_confidence["pct"].max() * 1.2]
)

fig.update_xaxes(showgrid=False)

fig.show()

## Insights

### **Insight 1 — Une forte part de mentions à faible confiance**

46% des mentions sur le Bénin sont classées en **Low confidence**. Une part importante de la couverture détectée par GDELT repose donc sur des signaux médiatiques jugés peu robustes par le système.

### **Insight 2 — Un noyau solide de mentions à haute confiance**

38% des mentions sont classées en **High confidence**. Le Bénin bénéficie donc aussi d'un volume important de couverture médiatique fortement consolidée dans les données GDELT.

### **Insight 3 — Une distribution très contrastée**

La catégorie **Medium** ne représente que 16,5% des mentions. La couverture du Bénin se répartit donc surtout entre signaux faibles et signaux forts, avec peu de mentions intermédiaires.

## Synthèse

En 2025, les mentions sur le Bénin dans GDELT présentent une structure **contrastée** : 46% de faible confiance, 38% de haute confiance et 16,5% de confiance moyenne. Cela suggère une visibilité médiatique à deux vitesses, partagée entre un socle solide de couverture bien captée et une part importante de signaux plus incertains.

## Question analytique

# Le Bénin est-il perçu positivement ou négativement dans les médias internationaux en 2025 ?

In [ ]:
# Ton éditorial des mentions GDELT au Bénin en 2025
query_tone = f"""
SELECT
  m.tone_category AS tone_cat,
  COUNT(*)        AS nb_mentions
FROM `{PROJECT_ID}.{DATASET_ID}.mentions_clean` AS m
JOIN `{PROJECT_ID}.{DATASET_ID}.events_clean`   AS e
  ON m.GLOBALEVENTID = e.GLOBALEVENTID
WHERE
  e.ActionGeo_CountryCode = 'BN'
  AND CAST(e.Year AS INT64) = 2025
  AND m.tone_category IS NOT NULL
GROUP BY tone_cat
ORDER BY nb_mentions DESC
"""

df_tone = charger_donnees(query_tone).copy()

Downloading: 100%|██████████|


In [29]:
import plotly.graph_objects as go

# Calcul des pourcentages
df_tone["pct"] = (
    df_tone["nb_mentions"] / df_tone["nb_mentions"].sum() * 100
).round(1)

# Harmoniser les libellés si besoin
df_tone["tone_cat"] = df_tone["tone_cat"].astype(str).str.strip().str.title()

# Tri décroissant par proportion
df_tone = df_tone.sort_values("pct", ascending=False)

# Couleurs selon le ton
colors_tone = {
    "Very Positive": "#3fa34d",
    "Positive": "#6daa45",
    "Neutral": "#e8af34",
    "Negative": "#d16363",
    "Very Negative": "#b23a48"
}

bar_colors = [colors_tone.get(c, "#b8c0c8") for c in df_tone["tone_cat"]]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_tone["tone_cat"],
    y=df_tone["pct"],
    marker_color=bar_colors,
    marker_line_color="rgba(255,255,255,0.10)",
    marker_line_width=1,
    text=df_tone["pct"].astype(str) + "%",
    textposition="outside",
    hovertemplate="%{x} : <b>%{y}%</b><extra></extra>"
))

fig.update_layout(
    title=dict(
        text="Ton éditorial des mentions (tone_category — MentionDocTone)",
        x=0.02,
        xanchor="left",
        y=0.97,
        yanchor="top",
        font=dict(size=18, color="#b8c0c8")
    ),
    template="plotly_dark",
    showlegend=False,
    xaxis_title="",
    yaxis_title="% des mentions",
    margin=dict(t=110, l=70, r=40, b=70),
    width=1100,
    height=520,
    bargap=0.22,
    barcornerradius=12
)

fig.add_annotation(
    text="<b>Bar chart</b>",
    xref="paper", yref="paper",
    x=0.98, y=1.08,
    showarrow=False,
    xanchor="right",
    yanchor="middle",
    font=dict(size=12, color="#4f98a3"),
    bgcolor="rgba(79,152,163,0.12)",
    bordercolor="#4f98a3",
    borderwidth=1,
    borderpad=6
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zeroline=False,
    range=[0, df_tone["pct"].max() * 1.2]
)

fig.update_xaxes(showgrid=False)

fig.show()

## Insights · Ton éditorial des mentions GDELT

### **Insight 1 — Un ton majoritairement négatif malgré une réalité coopérative**

Le ton éditorial des mentions montre que **56% des contenus médiatiques** sur le Bénin en 2025 sont négatifs ou très négatifs, contre seulement **28,3% de ton positif**. Cette perception contraste avec la nature réelle des événements, où la coopération et la communication diplomatique dominent largement (54% des événements).

### **Insight 2 — Le biais de négativité médiatique**

Les médias internationaux couvrent le Bénin de façon **2 fois plus négative que positive**, ce qui reflète un biais classique : les crises, conflits et menaces terroristes génèrent plus d'attention médiatique que les consultations diplomatiques et les accords de coopération. La violence (8,5% des événements) occupe donc une place disproportionnée dans le récit médiatique global.

### **Insight 3 — Un déficit de narration positive à combler**

Seulement **6,7% des mentions** portent un ton très positif, ce qui suggère que les réussites et initiatives positives du Bénin (coopération régionale, stabilité relative, investissements) sont sous-représentées dans l'écosystème médiatique international. Cette asymétrie narrative peut affecter la réputation et l'attractivité du pays.

## Synthèse · Ton éditorial des mentions GDELT

Le ton éditorial des mentions GDELT révèle une asymétrie narrative frappante : **56% des contenus** sur le Bénin en 2025 sont négatifs ou très négatifs, contre seulement **28% de ton positif**, alors même que la réalité événementielle est dominée par la coopération diplomatique (**54%**). Cette distorsion médiatique amplifie la visibilité des crises (violence terroriste, tensions) au détriment des dynamiques coopératives, créant un décalage entre la perception internationale et la réalité factuelle du pays.

## Question analytique

# Dans quelle mesure la visibilité médiatique du Bénin en 2025 est-elle façonnée par des contenus traduits versus non traduits ?

In [75]:
#  Part des mentions traduites vs non traduites · Bénin 2025
query_translated = f"""
SELECT
  m.is_translated AS is_translated,
  COUNT(*)        AS nb_mentions
FROM `{PROJECT_ID}.{DATASET_ID}.mentions_clean` AS m
JOIN `{PROJECT_ID}.{DATASET_ID}.events_clean`   AS e
  ON m.GLOBALEVENTID = e.GLOBALEVENTID
WHERE
  e.ActionGeo_CountryCode = 'BN'
  AND CAST(e.Year AS INT64) = 2025
  AND m.is_translated IS NOT NULL
GROUP BY is_translated
ORDER BY nb_mentions DESC
"""

df_translated = charger_donnees(query_translated).copy()

Downloading: 100%|██████████|


In [67]:
import plotly.graph_objects as go

# Remapper True/False en labels lisibles
df_translated["translated_label"] = df_translated["is_translated"].map({
    True: "Traduit (Translingual)",
    False: "Non traduit"
})

# Calcul des pourcentages
df_translated["pct"] = (
    df_translated["nb_mentions"] / df_translated["nb_mentions"].sum() * 100
).round(1)

# Tri décroissant
df_translated = df_translated.sort_values("pct", ascending=False)

# Couleurs : bleu pour traduit, gris pour non traduit
bar_colors = []
for v in df_translated["is_translated"]:
    if v:
        bar_colors.append("#4f98a3")  # traduit
    else:
        bar_colors.append("#7f8c8d")  # non traduit

fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_translated["translated_label"],
    y=df_translated["pct"],
    marker_color=bar_colors,
    marker_line_color="rgba(255,255,255,0.10)",
    marker_line_width=1,
    text=df_translated["pct"].astype(str) + "%",
    textposition="outside",
    hovertemplate="%{x} : <b>%{y}%</b><extra></extra>"
))

fig.update_layout(
    title=dict(
        text="Part des mentions traduites (is_translated)",
        x=0.02,
        xanchor="left",
        y=0.97,
        yanchor="top",
        font=dict(size=18, color="#b8c0c8")
    ),
    template="plotly_dark",
    showlegend=False,
    xaxis_title="",
    yaxis_title="% des mentions",
    margin=dict(t=110, l=70, r=40, b=70),
    width=900,
    height=480,
    bargap=0.35,
    barcornerradius=12
)

fig.add_annotation(
    text="<b>Bar chart</b>",
    xref="paper", yref="paper",
    x=0.98, y=1.08,
    showarrow=False,
    xanchor="right",
    yanchor="middle",
    font=dict(size=12, color="#4f98a3"),
    bgcolor="rgba(79,152,163,0.12)",
    bordercolor="#4f98a3",
    borderwidth=1,
    borderpad=6
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zeroline=False,
    range=[0, df_translated["pct"].max() * 1.3]
)

fig.update_xaxes(showgrid=False)

fig.show()

## Insights · Part des mentions traduites vs non traduites

### **Insight 1 — La visibilité repose principalement sur des contenus non traduits**

Environ **79,8% des mentions** (25 782 sur 32 308) proviennent de documents non traduits par le pipeline translingual de GDELT. Cela signifie que la couverture des événements liés au Bénin repose majoritairement sur des contenus directement exploitables par GDELT, sans passage par une traduction automatique.

### **Insight 2 — Une part non négligeable de contenus traduits**

Environ **20,2% des mentions** (6 526) sont issues du pipeline translingual, c'est-à-dire de contenus initialement rédigés dans d'autres langues puis traduits automatiquement. Les récits produits en français et dans d'autres langues contribuent donc à la visibilité internationale du Bénin, mais restent minoritaires dans le volume global de mentions.

### **Insight 3 — L'image du Bénin est structurée par les langues dominantes **

Combiné avec la visualisation sur `source_language` (où l'anglais domine), ce résultat montre que l'image internationale du Bénin est largement construite et relayée à partir de rédactions dans les langues principales de l'écosystème médiatique global. Les contenus produits dans d'autres langues sont intégrés au récit mondial via un processus de traduction automatique, ajoutant une couche supplémentaire de médiation technologique entre les sources locales et leur diffusion internationale.

## Synthèse · Mentions traduites vs non traduites

La variable `is_translated` révèle que près de **80% des mentions** liées au Bénin proviennent de documents non traduits, donc directement exploitables par GDELT dans leur langue d'origine. Seules **20,2% des mentions** passent par le pipeline translingual. En croisant ce résultat avec la répartition par langue source (dominée par l'anglais), on constate que la visibilité internationale du Bénin est structurée par les langues dominantes de l'écosystème médiatique global, les voix locales non anglophones étant présentes mais secondaires dans le volume total de mentions.

## Question analytique

# Quelle typologie d'événements domine la couverture médiatique du Bénin en 2025 ?

In [ ]:
# Top 19 EventCategory · Bénin 2025

query_eventcategory = f"""
SELECT
  EventCategory,
  COUNT(*) AS nb_occurrences
FROM `{PROJECT_ID}.{DATASET_ID}.events_clean`
WHERE
  EventCategory IS NOT NULL
GROUP BY
  EventCategory
ORDER BY
  nb_occurrences DESC
"""

df_eventcategory = charger_donnees(query_eventcategory).copy()
df_eventcategory.to_csv("eventcategory_benin_2025.csv", index=False, encoding="utf-8")

# Afficher le top
df_eventcategory

Downloading: 100%|██████████|


,EventCategory,nb_occurrences
0,Consult,8158
1,Make Public Statement,4474
2,Engage in Diplomatic Cooperation,4340
3,Appeal,2100
4,Fight,1771
5,Disapprove,1716
6,Coerce,1486
7,Express Intent to Cooperate,1265
8,Provide Aid,1065
9,Yield,988


In [73]:
import plotly.express as px


# Bar chart horizontal trié croissant pour lecture naturelle
fig_eventcat = px.bar(
    df_eventcategory.sort_values("nb_occurrences", ascending=True),
    x="nb_occurrences",
    y="EventCategory",
    orientation="h",
    text="nb_occurrences",
    title="Catégories d'événements GDELT au Bénin en 2025",
    labels={
        "nb_occurrences": "Nombre d'événements",
        "EventCategory": "Catégorie"
    },
    color="nb_occurrences",
    color_continuous_scale="RdYlGn"  # Rouge (conflit) → Jaune → Vert (coopération)
)


fig_eventcat.update_traces(
    textposition="outside",
    textfont=dict(size=13),
    hovertemplate="<b>%{y}</b><br>Événements : %{x:,}<extra></extra>"
)


fig_eventcat.update_layout(
    template="plotly_dark",
    height=900,
    width=1500,
    showlegend=False,
    coloraxis_showscale=False,
    title_x=0.02,
    title_font_size=20,
    margin=dict(l=20, r=140, t=80, b=50),
    font=dict(size=14, family="Inter, Arial, sans-serif")
)


fig_eventcat.update_xaxes(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.12)",
    title_font_size=14
)


fig_eventcat.update_yaxes(
    showgrid=False,
    title_font_size=14
)

# Enregistrement de la visualisation en fichier HTML interactif
fig_eventcat.write_html("categories_evenements_gdelt_benin_2025.html")

fig_eventcat.show()

## Insight 1 : Le Bénin s'affirme comme hub diplomatique régional démesuré

L'analyse des 31,504 événements GDELT révèle une réalité frappante : 54% de l'activité du Bénin en 2025 relève de la communication diplomatique, avec 16,972 événements de consultations, déclarations publiques et coopération formelle.

## Insight 2 : Une menace sécuritaire réelle mais géographiquement contenue

Les 2,307 actes de violence physique recensés (1,771 combats + 536 agressions) confirment que le Bénin subit la contagion djihadiste sahélienne, notamment dans le nord du pays (Alibori, Atacora). Cependant, cette violence reste 5 fois moins fréquente que les tensions verbales (5,134 événements de désapprobations, menaces et rejets). La violence est réelle, mais elle ne domine pas encore l'agenda national, contrairement à des cas réels dans d’autres pays où elle structure toute la vie politique.

## Insight 3 : Un écart préoccupant entre coopération affichée et coopération concrète

Sur les 6,104 événements de coopération identifiés, seulement 686 relèvent de l'aide matérielle (11%), le reste étant constitué d'intentions, d'appels à l'aide ou de concessions symboliques. Ce décalage révèle que le Bénin multiplie les consultations et les annonces de partenariats, mais reçoit peu d'aide concrète en retour. Cette dynamique traduit une dépendance structurelle aux promesses internationales et une fragilité budgétaire : le pays mise sur sa réputation diplomatique pour attirer des financements, mais l'essentiel reste au stade de l'engagement verbal.

## Synthèse finale

En 2025, l'actualité du Bénin est dominée par la communication institutionnelle et les consultations diplomatiques (54% des événements), suivies par des formes variées de coopération (19%) et de tensions verbales (16%). La violence armée, bien que réelle avec 2,307 événements de combats et agressions, reste minoritaire (8.5%) et concentrée dans le nord du pays.

In [32]:
query_lang = f"""
SELECT source_language, tone_category, COUNT(*) as total
FROM `{PROJECT_ID}.{DATASET_ID}.gkg_clean`
WHERE source_language IS NOT NULL
GROUP BY source_language, tone_category
ORDER BY total DESC
LIMIT 20
"""

df_lang = charger_donnees(query_lang)

fig = px.bar(
    df_lang,
    x="source_language",
    y="total",
    color="tone_category",
    barmode="stack",
    title="Sentiment par Langue Source des Articles au Bénin (2025)"
)
fig.update_layout(template="plotly_dark")
fig.show()

## 01 — L'anglais domine la narration d'un pays francophone

**~25 000 articles** en anglais contre **~12 000** en français — soit **2× plus** de couverture anglophone.

Le français béninois produit peu de contenu numérique indexé à l'échelle mondiale.
Le Bénin compte ~100 journaux et ~200 médias audiovisuels *(RSF, 2025)*, mais leur
diffusion reste locale. La francophonie béninoise parle à voix basse sur l'internet
mondial — et GDELT enregistre ce silence.

---

## 02 — Le sentiment négatif est structurel, pas conjoncturel

Dans chaque barre — `eng`, `fra`, `spa`, `por` — le **Négatif** et le **Très Négatif**
occupent la majeure partie de la hauteur, toutes langues confondues.

Ce pattern uniforme transcende les cultures éditoriales. En 2023–2025, le Bénin a connu
la suspension de médias indépendants et des restrictions du Code du numérique
*(Amnesty International, 2025)*. Le sentiment négatif dans les données est le reflet
d'une **presse sous tension**, pas une distorsion algorithmique.

---

## 03 — Neuf langues présentes, mais 50 invisibles

Les codes `eng`, `fra`, `spa`, `ita`, `por`, `rus`, `ell`, `bul`, `zho`, `ara`
représentent la quasi-totalité des articles.

Pourtant, le Bénin compte **plus de 50 langues vivantes** — fon, yoruba, bariba, dendi…
Ces langues, portées par la radio et la tradition orale, ne génèrent pas de texte
numérique indexable. GDELT ne capte pas la réalité vernaculaire du pays.

> **L'absence d'une langue dans les données n'est pas son silence — c'est son exclusion.**